# 🧪 Assignment 1 - Color-Based Object Detection and Analysis

## Objective and steps

In this assignment, you will identify and analyze objects in the image `shapes&color.png` based on their color. You have to apply the techniques learned during Lab 1. The colors to segment are: red, blue, yellow and green.

![image](\Images4notebook\shapes&colors.png)

Complete the provided notebook. Each section contains:
- Questions that you must answer to summarize your findings
- Key ideas that will help guide your implementation and analysis

### 📌 Step 1 – Color and Filter Comparison

1. The **original image**
2. The filtered image(s) obtained using the filters:
    - Averaging
    - Gaussian
3. Compare how filtering affects color detection performance

**📤 Output 1**

Produce one figure showing the comparison between:
- Color masks computed on the **orginal image**
- Color masks computed on the **filtered image(s)**

### 📌 Step 2 – Bounding Box Detection

Using the color masks chosen at the previous step:

1. Detect the objects corresponding to each color
2. Draw a box around each detected object

**📤 Output 2**

Produce one figure with 4 subplots, showing the boxes over the images. One subplot for each color.

### 📌 Step 3 – Object Area Extraction

For each detected object:

1. Extract the actual object area (not just the bounding box)
2. Display only the pixels belonging to the detected objects of that color

**📤 Output 3**

Produce one figure with 4 subplots, where each subplot shows only the detected objects for each color.

## 🛠️ Requirements and environment setup

Import all the necessary libraries and paths

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt


MATERIAL_DIR = 'material'
SAVE_DIR = 'saves'

IMG_NAME = 'shapes&colors'
PATH_IMG = os.path.join(MATERIAL_DIR, IMG_NAME)

## 📌 Step 1 – Color and Filter Comparison

- Compute color masks on the original image
- Apply Filtering (2 averaging box of different sizes and a Gaussian filter)
- Display Comparison

In [ ]:
def show_bgr(img_bgr, title=None):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    if title:
        plt.title(title)
    plt.axis('off')

def show_gray(img_gray, title=None):
    plt.imshow(img_gray, cmap='gray')
    if title:
        plt.title(title)
    plt.axis('off')

def overlay_mask_on_bgr(img_bgr, mask_u8, color_bgr, alpha=0.45):
    color_layer = np.zeros_like(img_bgr)
    color_layer[mask_u8 > 0] = color_bgr
    return cv2.addWeighted(img_bgr, 1-alpha, color_layer, alpha, 0)
    
img = cv2.imread(PATH_IMG)
assert img is not None, f'Could not read image at: {PATH_IMG}'

print('Shape (H,W,C):', img.shape)
print('Dtype:', img.dtype)

plt.figure(figsize=(10,6))
show_bgr(img, 'Input image')
plt.show()

### 📊 Compute Hystograms for original image

**Histograms** help you understand pixel intensity distributions in each channel. 

This is useful for: 
1. Choosing threshold values for segmentation understanding 
2. How filtering changes intensities

In [ ]:
b, g, r = cv2.split(img)

hist_b = cv2.calcHist([b], [0], None, [256], [0, 256])
hist_g = cv2.calcHist([g], [0], None, [256], [0, 256])
hist_r = cv2.calcHist([r], [0], None, [256], [0, 256])

plt.figure(figsize=(8,10))

plt.subplot(3,1,1)
plt.plot(hist_r, color='r')
plt.title('Red channel histogram')
plt.xlabel('Pixel intensity')
plt.ylabel('Frequency')

plt.subplot(3,1,2)
plt.plot(hist_g, color='g')
plt.title('Green channel histogram')
plt.xlabel('Pixel intensity')
plt.ylabel('Frequency')

plt.subplot(3,1,3)
plt.plot(hist_b, color='b')
plt.title('Blue channel histogram')
plt.xlabel('Pixel intensity')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

We observe:
- A very large peak close to intensity 255 in all three channels (Red, Green, Blue).
- The rest of the histogram values are extremely small compared to that peak.

This means that a large portion of the image has very high values in all three channels; so a significant number of pixels are white or nearly white. *The background dominates the histogram distribution.* Because histograms represent the global distribution of pixel intensities, the dominant class in the image (the white background) heavily influences the shape of the histogram.

A more reliable strategy is to calculate histograms only on no-white pixels

In [ ]:
mask_not_white = (r < 240) | (g < 240) | (b < 240)
mask_not_white_u8 = mask_not_white.astype(np.uint8) * 255

hist_r = cv2.calcHist([r], [0], mask_not_white_u8, [256], [0, 256])
hist_g = cv2.calcHist([g], [0], mask_not_white_u8, [256], [0, 256])
hist_b = cv2.calcHist([b], [0], mask_not_white_u8, [256], [0, 256])

plt.plot(hist_r, label="Red (no white)")
plt.plot(hist_g, label="Green (no white)")
plt.plot(hist_b, label="Blue (no white)")
plt.legend()
plt.show()

Now this histogram is much more informative. You can clearly see how the colored objects distribute their intensities across the three channels. From this plot you can extract practical rules for segmentation.

**🔴 Red Objects**

The red channel $R$ shows strong peaks at high intensities (around 200–255), while the green and blue channels are significantly lower in those regions. This suggests that red objects can be segmented by selecting pixels where:

- R is high (for example $R > 180$ or $R > 200$),

- and R is significantly greater than both G and B (for example $R − G > 40$ and $R − B > 40$).

This ensures selecting pixels where red is dominant, not just bright.

**🟢 Green Objects**

The green channel $G$ has a strong peak around mid-to-high intensities (roughly 180–220), while red and blue are lower in that region. This suggests a rule such as:

- $G > 160$ or $G > 170$
- $G − R > 30$ or $G− B > 30$

**🔵 Blue Objects**

The blue channel $B$ shows clear peaks at moderate intensities (for example around 40–60) and also some higher values. In those regions, red and green are lower. So blue segmentation should require:

- $B > 50$ or $B > 100$,
- $B − R > 30$ or $B − G > 30$

**🟡 Yellow Objects**

For yellow objects, the key observation is that yellow appears when both red and green are high simultaneously, while blue remains low. In the histogram, you see overlapping peaks for red and green in the higher range (around 180–220), while blue is relatively small there. Therefore:

- $R > 160$
- $G > 160$
- $B < 120$ (or another value clearly below the R/G peaks).


In [ ]:
B, G, R = b, g, r

# --- Example thresholds (TUNE THESE!) ---
# Red objects (RGB baseline)
mask_red_rgb = (R > 150) & (G < 110) & (B < 110)
# Blue objects
mask_blue_rgb = (B > 150) & (G < 170) & (R < 190)
# Yellow objects (high R and G, low B)
mask_yellow_rgb = (R > 160) & (G > 160) & (B < 150)
# Green objects
mask_green_rgb = (G > 110) & (R < 180) & (B < 160)

# Convert to uint8 masks
mask_red_rgb_u8 = (mask_red_rgb.astype(np.uint8) * 255)
mask_blue_u8 = (mask_blue_rgb.astype(np.uint8) * 255)
mask_yellow_u8 = (mask_yellow_rgb.astype(np.uint8) * 255)
mask_green_u8 = (mask_green_rgb.astype(np.uint8) * 255)

plt.figure(figsize=(12,8))
plt.subplot(2,3,1); show_bgr(img, 'Original')
plt.subplot(2,3,2); show_gray(mask_red_rgb_u8, 'Mask: Red (RGB)')
plt.subplot(2,3,3); show_gray(mask_blue_u8, 'Mask: Blue (RGB)')
plt.subplot(2,3,5); show_gray(mask_yellow_u8, 'Mask: Yellow (RGB)')
plt.subplot(2,3,6); show_gray(mask_green_u8, 'Mask: Green (RGB)')
plt.tight_layout(); plt.show()

In [ ]:
ov_red_rgb = overlay_mask_on_bgr(img, mask_red_rgb_u8, (0,0,255))
ov_blue = overlay_mask_on_bgr(img, mask_blue_u8, (255,0,0))
ov_yellow = overlay_mask_on_bgr(img, mask_yellow_u8, (0,255,255))
ov_green = overlay_mask_on_bgr(img, mask_green_u8, (0,255,0))

plt.figure(figsize=(14,8))
plt.subplot(2,2,1); show_bgr(ov_red_rgb, 'Overlay: Red (RGB)')
plt.subplot(2,2,2); show_bgr(ov_blue, 'Overlay: Blue')
plt.subplot(2,2,3); show_bgr(ov_yellow, 'Overlay: Yellow')
plt.subplot(2,2,4); show_bgr(ov_green, 'Overlay: Green')
plt.tight_layout(); plt.show()

###  🏻 Apply Filtering and Recompute color masks 🎭

Filtering can reduce noise and smooth intensity variations. This may improve segmentation, but it can also blur edges and reduce contrast.

We implement 3 different filters:

In [ ]:
img_blur_3  = cv2.blur(img, (3,3))
img_blur_7  = cv2.blur(img, (7,7))
img_gauss_5 = cv2.GaussianBlur(img, (5,5), 0)

variants = {
    "Original": img,
    "Box 3x3": img_blur_3,
    "Box 7x7": img_blur_7,
    "Gaussian": img_gauss_5
}

Now we have to recompute the histograms and masks for each variant. For convenience, we define the following functions.

In [ ]:
def compute_rgb_histograms(img_bgr):
    b, g, r = cv2.split(img_bgr)
    hist_b = cv2.calcHist([b], [0], mask_not_white_u8, [256], [0, 256])
    hist_g = cv2.calcHist([g], [0], mask_not_white_u8, [256], [0, 256])
    hist_r = cv2.calcHist([r], [0], mask_not_white_u8, [256], [0, 256])
    return hist_r, hist_g, hist_b


def compute_color_masks_rgb(img_bgr):
    """Apply the SAME RGB threshold rules to img_bgr."""
    b, g, r = cv2.split(img_bgr)
    B, G, R = b, g, r

    mask_red    = (R > 150) & (G < 110) & (B < 110)
    mask_blue   = (B > 150) & (G < 170) & (R < 190)
    mask_yellow = (R > 160) & (G > 160) & (B < 150)
    mask_green  = (G > 110) & (R < 180) & (B < 160)

    return {
        "Red":    (mask_red.astype(np.uint8) * 255),
        "Blue":   (mask_blue.astype(np.uint8) * 255),
        "Yellow": (mask_yellow.astype(np.uint8) * 255),
        "Green":  (mask_green.astype(np.uint8) * 255),
    }

hists_by_variant = {}
masks_by_variant = {}

for name, im in variants.items():
    hists_by_variant[name] = compute_rgb_histograms(im)
    masks_by_variant[name] = compute_color_masks_rgb(im)

fig, ax = plt.subplots(2, 2, figsize=(14, 8))
axs = ax.ravel()

for k, name in enumerate(variants.keys()):
    hr, hg, hb = hists_by_variant[name]
    axs[k].plot(hr, label='R')
    axs[k].plot(hg, label='G')
    axs[k].plot(hb, label='B')
    axs[k].set_title(f'RGB histograms — {name}')
    axs[k].set_xlim(0, 255)
    axs[k].legend()

plt.tight_layout()
plt.show()

colors = ["Red", "Blue", "Yellow", "Green"]
variant_names = list(variants.keys())

fig, ax = plt.subplots(len(colors), len(variant_names), figsize=(16, 10))

for i, c in enumerate(colors):
    for j, v in enumerate(variant_names):
        m = masks_by_variant[v][c]
        ax[i, j].imshow(m, cmap="gray")
        if i == 0:
            ax[i, j].set_title(v)
        if j == 0:
            ax[i, j].set_ylabel(c, rotation=0, labelpad=30, va="center")
        ax[i, j].axis("off")

plt.tight_layout()


**🎯 Output 1 — Mission Accomplished!**

| Filtering Method     | Noise Level                                     | Edge Sharpness     | Small Details                         | Object Shape Integrity             | Summary              |
| -------------------- | ----------------------------------------------- | ------------------ | ------------------------------------- | ---------------------------------- | ------------------------------- |
| Original (No Filter) | Higher noise (isolated pixels, small fragments) | Very sharp         | Fully preserved                       | Good but noisy boundaries          | Good detail, but unstable masks |
| Box 3×3              | Slightly reduced noise                          | Mostly preserved   | Largely preserved                     | Compact and cleaner masks          | Good compromise                 |
| Box 7×7              | Strong noise reduction                          | Noticeably blurred | Some thin structures degraded or lost | Slight erosion of small components | Too aggressive                  |
| Gaussian 5×5         | Reduced noise                                   | Well preserved     | Preserved better than Box 7×7         | Natural-looking, stable masks      | Best trade-off                  |

Moderate smoothing improves segmentation stability by reducing small spurious detections and fragmented regions. However, excessive smoothing (Box 7×7) begins to remove relevant high-frequency information, degrading thin structures and small components. Gaussian 5×5 provides the best balance between noise reduction and shape preservation, making it the most suitable filtering choice for subsequent steps such as connected component analysis and bounding box extraction.

![image](\Images4notebook\Filtering.png)

## 📌 Step 2 – Bounding Box Detection

Using the color masks chosen at the previous step:

1. Use morphological operations to improve the quality of the chosen masks
1. Detect the objects corresponding to each color
2. Draw a box around each detected object

### 💅 Refine Using Morphological Operations

Morphological operations help regularize the mask structure before proceeding to connected component analysis and bounding box extraction.

In [ ]:
mask_red_u8 = masks_by_variant["Gaussian"]["Red"]
mask_blue_u8 = masks_by_variant["Gaussian"]["Blue"]
mask_yellow_u8 = masks_by_variant["Gaussian"]["Yellow"]
mask_green_u8 = masks_by_variant["Gaussian"]["Green"]

After selecting the best filtering variant (Gaussian 5×5), we further improve the binary masks using morphological operations. We define a square structuring element (kernel) of size 5×5. The size of the kernel determines how strong the morphological effect will be. A larger kernel removes more noise but may also distort object shapes.

In [ ]:
kernel = np.ones((5,5), np.uint8)

def refine(mask_u8, kernel=kernel):
    opened = cv2.morphologyEx(mask_u8, cv2.MORPH_OPEN, kernel)
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
    return opened, closed

We define a new function `refine` that applies two consecutive operations:
1. Opening (erosion followed by dilation)
    - Removes small isolated foreground pixels
    - Eliminates thin noise fragments
2. Closing (dilation followed by erosion)
    - Fills small holes inside objects
    - Makes object regions more compact

In [ ]:
red_open, red_ref = refine(mask_red_u8)
blue_open, blue_ref = refine(mask_blue_u8)
yellow_open, yellow_ref = refine(mask_yellow_u8)
green_open, green_ref = refine(mask_green_u8)

plt.figure(figsize=(14,8))
plt.subplot(2,4,1); show_gray(mask_red_u8, 'Red raw')
plt.subplot(2,4,2); show_gray(red_ref, 'Red refined')
plt.subplot(2,4,3); show_gray(mask_blue_u8, 'Blue raw')
plt.subplot(2,4,4); show_gray(blue_ref, 'Blue refined')

plt.subplot(2,4,5); show_gray(mask_yellow_u8, 'Yellow raw')
plt.subplot(2,4,6); show_gray(yellow_ref, 'Yellow refined')
plt.subplot(2,4,7); show_gray(mask_green_u8, 'Green raw')
plt.subplot(2,4,8); show_gray(green_ref, 'Green refined')
plt.tight_layout(); plt.show()

### 📦 Bounding Box detection and drawing

Now, we can detect individual objects and draw a bounding box around each one. A practical and robust way to do this is to treat each connected foreground region in the binary mask as a candidate object and compute a bounding box from its pixel extent.

In this solution we use `cv2.connectedComponentsWithStats`, which returns, for each connected component, a set of statistics including its bounding box coordinates and its area. More information can be found [here](https://docs.opencv.org/3.4/d3/dc0/group__imgproc__shape.html#ga107a78bf7cd25dec05fb4dfc5c9e765f)

We define the `draw_bboxes_from_mask` function that implements the following steps:
1. Convert the mask into a binary image (0/1).
2. Compute connected components and retrieve their statistics.
3. For each component, filter out small regions using a minimum area threshold to remove residual noise.
4. Draw a rectangle using the bounding box coordinates.

In [ ]:
def draw_bboxes_from_mask(img_bgr, mask_u8, box_color=(0,255,0), min_area=500, thickness=2):
    out = img_bgr.copy()
    binary = (mask_u8 > 0).astype(np.uint8)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)

    for i in range(1, num_labels):  # 0 = background
        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]

        if area < min_area:
            continue

        cv2.rectangle(out, (x, y), (x + w, y + h), box_color, thickness)

    return out

A key hyperparameter here is min_area. The refined masks can still contain very small connected components caused by noise or thresholding artifacts. These should not produce bounding boxes, because they would create false detections. The value of min_area should be chosen so that it removes small blobs while preserving real objects.

In [ ]:
# Tune this: it should remove tiny blobs without deleting real objects
MIN_AREA_BBOX = 800

img_red_bbox    = draw_bboxes_from_mask(img, red_ref,    box_color=(0,0,255),   min_area=MIN_AREA_BBOX)
img_blue_bbox   = draw_bboxes_from_mask(img, blue_ref,   box_color=(255,0,0),   min_area=MIN_AREA_BBOX)
img_yellow_bbox = draw_bboxes_from_mask(img, yellow_ref, box_color=(0,255,255), min_area=MIN_AREA_BBOX)
img_green_bbox  = draw_bboxes_from_mask(img, green_ref,  box_color=(0,255,0),   min_area=MIN_AREA_BBOX)

fig, axs = plt.subplots(2, 2, figsize=(14, 10))

axs[0,0].imshow(cv2.cvtColor(img_red_bbox, cv2.COLOR_BGR2RGB))
axs[0,0].set_title('Red objects')
axs[0,0].axis('off')

axs[0,1].imshow(cv2.cvtColor(img_blue_bbox, cv2.COLOR_BGR2RGB))
axs[0,1].set_title('Blue objects')
axs[0,1].axis('off')

axs[1,0].imshow(cv2.cvtColor(img_yellow_bbox, cv2.COLOR_BGR2RGB))
axs[1,0].set_title('Yellow objects')
axs[1,0].axis('off')

axs[1,1].imshow(cv2.cvtColor(img_green_bbox, cv2.COLOR_BGR2RGB))
axs[1,1].set_title('Green objects')
axs[1,1].axis('off')

plt.tight_layout()
plt.show()


**🎯 Output 2 — Mission Accomplished!**

## 📌 Step 3 – Object Area Extraction

1. Extract the actual object area (not just the bounding box)
2. Display only the pixels belonging to the detected objects of that color

### 📐 Compute area from masks

At this stage we want to extract the actual pixels belonging to each detected object (not the bounding box region). The simplest approach would be to apply the refined mask directly to the original image. However, masks can contain small holes or fragmented regions, and edge-based contours can be broken for some objects. To make the extraction stable, we use the color mask as a “seed” to guarantee a filled region, and optionally compute Sobel edges only to draw a nicer contour.

The pipeline for each connected component is:

1. Find connected components in the refined color mask
2. For each object ROI (a smaller sub-region of the image that contains only the part we want to analyze):
    - Build a filled object region from the color seed mask (with a small morphological close).
    - Optionally compute Sobel edges inside the ROI to draw an edge-like contour (fallback to seed contour if edges are broken).

In [ ]:
def areas_from_masks(
    img_bgr,
    mask_u8,
    min_area=800,
    pad=8,
    seed_close_ks=5,
    seed_close_iters=1,
    use_sobel_for_drawing=True,
    blur_ks=5,
    sobel_ksize=3,
    edge_close_ks=7,
    edge_close_iters=2
):
    """
    For each connected component in mask_u8 (after min_area filtering), compute:
      - img_contours: black image with white contours for each object.
      - img_cutout: black image where only pixels belonging to objects are visible.

    Notes:
    - The seed region is derived from the color mask and is filled to avoid losing parts.
    - Sobel edges are used only for drawing a more 'edge-like' contour; if edges are
      broken/small, we fallback to the seed contour.
    """

    binary = (mask_u8 > 0).astype(np.uint8)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)

    # We prepare the two empty outputs (all black).
    img_contours = np.zeros_like(img_bgr)
    img_cutout   = np.zeros_like(img_bgr)

    H, W = img_bgr.shape[:2]

    Kseed = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (seed_close_ks, seed_close_ks))
    Kedge = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (edge_close_ks, edge_close_ks))

    

    for i in range(1, num):  # 0 is background
        # Calculate BBOX and Area
        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]

        if area < min_area:
            continue

        # Use BBOX to create ROI. This ROI contains only the current object (plus a small margin).
        x0 = max(0, x - pad); y0 = max(0, y - pad)
        x1 = min(W, x + w + pad); y1 = min(H, y + h + pad)

        roi = img_bgr[y0:y1, x0:x1]

        # 1) Edge maps can be broken or incomplete, so we do not rely on Sobel to decide which pixels belong to the object.
        # Instead, we use the color mask inside the ROI as a “seed region”:
        
        seed = (mask_u8[y0:y1, x0:x1] > 0).astype(np.uint8) * 255
        seed = cv2.morphologyEx(seed, cv2.MORPH_CLOSE, Kseed, iterations=seed_close_iters)

        # From this seed mask we extract the main contour:
        cnts_seed, _ = cv2.findContours(seed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts_seed:
            continue
        c_seed = max(cnts_seed, key=cv2.contourArea)

        # Filled mask of the object in ROI coordinates
        mask_roi = np.zeros_like(seed)
        cv2.drawContours(mask_roi, [c_seed], -1, 255, thickness=-1)

        # 2) Sobel contour (optional, for drawing only)
        c_draw = c_seed
        if use_sobel_for_drawing:
            gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
            gray = cv2.GaussianBlur(gray, (blur_ks, blur_ks), 0)

            gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=sobel_ksize)
            gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=sobel_ksize)
            mag = cv2.magnitude(gx, gy)

            mag_u8 = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

            # Otsu threshold on gradient magnitude
            _, edges = cv2.threshold(mag_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, Kedge, iterations=edge_close_iters)

            cnts_edge, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if cnts_edge:
                c_edge = max(cnts_edge, key=cv2.contourArea)
                # If the edge contour is too small/broken, fallback to seed contour
                if cv2.contourArea(c_edge) > 0.2 * cv2.contourArea(c_seed):
                    c_draw = c_edge

        # Draw contour in global coordinates
        c_global = c_draw + np.array([[x0, y0]], dtype=c_draw.dtype)
        cv2.drawContours(img_contours, [c_global], -1, (255, 255, 255), 2)

        # Stable cutout: copy only pixels inside filled seed region
        roi_out = img_cutout[y0:y1, x0:x1]
        roi_out[mask_roi > 0] = roi[mask_roi > 0]

    return img_contours, img_cutout


In [ ]:
red_cont, red_obj     = areas_from_masks(img, red_ref)
blue_cont, blue_obj   = areas_from_masks(img, blue_ref)
yellow_cont, yellow_obj = areas_from_masks(img, yellow_ref)
green_cont, green_obj = areas_from_masks(img, green_ref)

fig, ax = plt.subplots(2,2, figsize=(14,10))

ax[0,0].imshow(cv2.cvtColor(red_cont, cv2.COLOR_BGR2RGB))
ax[0,0].set_title("Red - contours")
ax[0,0].axis("off")

ax[0,1].imshow(cv2.cvtColor(blue_cont, cv2.COLOR_BGR2RGB))
ax[0,1].set_title("Blue - contours")
ax[0,1].axis("off")

ax[1,0].imshow(cv2.cvtColor(yellow_cont, cv2.COLOR_BGR2RGB))
ax[1,0].set_title("Yellow - contours")
ax[1,0].axis("off")

ax[1,1].imshow(cv2.cvtColor(green_cont, cv2.COLOR_BGR2RGB))
ax[1,1].set_title("Green - contours")
ax[1,1].axis("off")

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(2,2, figsize=(14,10))

ax[0,0].imshow(cv2.cvtColor(red_obj, cv2.COLOR_BGR2RGB))
ax[0,0].set_title("Red - inside contour")
ax[0,0].axis("off")

ax[0,1].imshow(cv2.cvtColor(blue_obj, cv2.COLOR_BGR2RGB))
ax[0,1].set_title("Blue - inside contour")
ax[0,1].axis("off")

ax[1,0].imshow(cv2.cvtColor(yellow_obj, cv2.COLOR_BGR2RGB))
ax[1,0].set_title("Yellow - inside contour")
ax[1,0].axis("off")

ax[1,1].imshow(cv2.cvtColor(green_obj, cv2.COLOR_BGR2RGB))
ax[1,1].set_title("Green - inside contour")
ax[1,1].axis("off")

plt.tight_layout()
plt.show()

**🎯 Output 3 — Mission Accomplished!**

The final results show that each object has been successfully segmented, refined, and extracted based on its color.